# Condition Occurrence — Allscripts Sunrise (SCM)

**OMOP CDM v5.4 — `condition_occurrence` table**

### Source Tables
- `_exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientdocdetail_bkp` — coded diagnoses (ICD/SNOMED)
- `_exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientdocumentcur` — parent document (dates, provider, visit linkage)

### Target
- Silver: `_exponent.omop_silver.condition_occurrence`
- Gold: `_exponent.omop.condition_occurrence`

### Strategy
- JOIN detail → document on `ClientDocumentGUID = doc.GUID`
- Filter to rows with valid `CodingScheme` / `CodingSchemeCode` (ICD-10-CM, ICD-9-CM, SNOMED)
- Map source codes to OMOP `condition_source_concept_id` via `concept` table
- Resolve standard `condition_concept_id` via `concept_relationship` (Maps to)
- Date: `COALESCE(doc.AuthoredDtm, doc.ServiceDtmUTC, doc.Entered, detail.CreatedWhen)`
- `condition_type_concept_id = 32817` (EHR encounter record)

### Dependencies
- `source_to_person` mapping must be populated for Allscripts SCM patients
- OMOP vocabulary tables (`concept`, `concept_relationship`) must be loaded

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Load Source Data
Load both source tables and join them on `detail.ClientDocumentGUID = doc.GUID`.

In [ ]:
detail_table = "_exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientdocdetail_bkp"
doc_table = "_exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientdocumentcur"

df_detail = spark.table(detail_table)
df_doc = spark.table(doc_table)

detail_count = df_detail.count()
doc_count = df_doc.count()

print(f"Detail rows (dbo_cv3clientdocdetail_bkp): {detail_count:,}")
print(f"Document rows (dbo_cv3clientdocumentcur): {doc_count:,}")

# Join detail to parent document
df_joined = df_detail.alias("detail").join(
    df_doc.alias("doc"),
    F.col("detail.ClientDocumentGUID") == F.col("doc.GUID"),
    "left"
)

joined_count = df_joined.count()
print(f"Joined rows: {joined_count:,}")
print(f"Distinct patients (detail.ClientGUID): {df_detail.select('ClientGUID').distinct().count():,}")

## EDA: CodingScheme Distribution
What coding systems are present? This determines vocabulary mapping logic.

In [ ]:
print("=== CodingScheme distribution ===")
display(
    df_detail.groupBy("CodingScheme")
    .agg(
        F.count("*").alias("count"),
        F.round(F.count("*") / F.lit(detail_count) * 100, 2).alias("pct")
    )
    .orderBy(F.desc("count"))
)

print("\n=== NULL / empty CodingScheme ===")
display(
    df_detail.select(
        F.sum(F.when(F.col("CodingScheme").isNull(), 1).otherwise(0)).alias("null_coding_scheme"),
        F.sum(F.when(F.trim(F.col("CodingScheme")) == "", 1).otherwise(0)).alias("empty_coding_scheme"),
        F.sum(F.when(F.col("CodingSchemeCode").isNull(), 1).otherwise(0)).alias("null_coding_scheme_code"),
        F.sum(F.when(F.trim(F.col("CodingSchemeCode")) == "", 1).otherwise(0)).alias("empty_coding_scheme_code"),
    )
)

## EDA: CodingSchemeCode Samples
For each `CodingScheme`, show 10 sample codes to validate they look like real diagnosis codes.

In [ ]:
coding_schemes = (
    df_detail.filter(F.col("CodingScheme").isNotNull())
    .select("CodingScheme")
    .distinct()
    .collect()
)

for row in coding_schemes:
    scheme = row["CodingScheme"]
    print(f"\n=== {scheme} — sample codes ===")
    display(
        df_detail.filter(F.col("CodingScheme") == scheme)
        .select("CodingSchemeCode", "KTreeItemName")
        .distinct()
        .limit(10)
    )

## EDA: HeadingOne Distribution
Identify heading values that represent diagnosis sections (e.g., "Diagnoses", "Problem List").

This informs filtering and `condition_status_source_value` mapping.

In [ ]:
display(
    df_detail.filter(F.col("CodingScheme").isNotNull())
    .groupBy("HeadingOne")
    .agg(
        F.count("*").alias("count"),
        F.round(
            F.count("*")
            / F.lit(
                df_detail.filter(F.col("CodingScheme").isNotNull()).count()
            )
            * 100,
            2,
        ).alias("pct"),
    )
    .orderBy(F.desc("count"))
)

## EDA: Document Filters
Distributions for document-level fields to determine valid filter criteria.

In [ ]:
# Scope to rows that have a coded diagnosis
df_coded = df_joined.filter(
    F.col("detail.CodingScheme").isNotNull()
    & F.col("detail.CodingSchemeCode").isNotNull()
)
coded_count = df_coded.count()

print(f"Rows with CodingScheme + CodingSchemeCode: {coded_count:,}")

print("\n=== doc.DocumentName distribution (top 30) ===")
display(
    df_coded.groupBy(F.col("doc.DocumentName"))
    .agg(
        F.count("*").alias("count"),
        F.round(F.count("*") / F.lit(coded_count) * 100, 2).alias("pct")
    )
    .orderBy(F.desc("count"))
    .limit(30)
)

print("\n=== doc.Status distribution ===")
display(
    df_coded.groupBy(F.col("doc.Status"))
    .agg(
        F.count("*").alias("count"),
        F.round(F.count("*") / F.lit(coded_count) * 100, 2).alias("pct")
    )
    .orderBy(F.desc("count"))
)

print("\n=== doc.Active distribution ===")
display(
    df_coded.groupBy(F.col("doc.Active"))
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
)

print("\n=== detail.Active distribution ===")
display(
    df_coded.groupBy(F.col("detail.Active"))
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
)

print("\n=== doc.IsCanceled distribution ===")
display(
    df_coded.groupBy(F.col("doc.IsCanceled"))
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
)

## EDA: Date Coverage
Determine the best date field for `condition_start_date`.

Fallback: `COALESCE(doc.AuthoredDtm, doc.ServiceDtmUTC, doc.Entered, detail.CreatedWhen)`

In [ ]:
date_fields = [
    ("doc.AuthoredDtm", "doc_AuthoredDtm"),
    ("doc.ServiceDtmUTC", "doc_ServiceDtmUTC"),
    ("doc.Entered", "doc_Entered"),
    ("detail.CreatedWhen", "detail_CreatedWhen"),
]

date_stats = df_coded.select(
    F.lit(coded_count).alias("total_coded_rows"),
    *[
        expr
        for col_path, alias in date_fields
        for expr in [
            F.count(F.col(col_path)).alias(f"{alias}_non_null"),
            F.round(
                F.count(F.col(col_path)) / F.lit(coded_count) * 100, 2
            ).alias(f"{alias}_pct"),
            F.min(F.col(col_path)).alias(f"{alias}_min"),
            F.max(F.col(col_path)).alias(f"{alias}_max"),
        ]
    ]
)

display(date_stats)

# Check how many rows the full COALESCE covers
coalesce_coverage = df_coded.select(
    F.count("*").alias("total"),
    F.sum(
        F.when(
            F.coalesce(
                F.col("doc.AuthoredDtm"),
                F.col("doc.ServiceDtmUTC"),
                F.col("doc.Entered"),
                F.col("detail.CreatedWhen"),
            ).isNotNull(),
            1,
        ).otherwise(0)
    ).alias("coalesce_non_null"),
    F.sum(
        F.when(
            F.coalesce(
                F.col("doc.AuthoredDtm"),
                F.col("doc.ServiceDtmUTC"),
                F.col("doc.Entered"),
                F.col("detail.CreatedWhen"),
            ).isNull(),
            1,
        ).otherwise(0)
    ).alias("coalesce_all_null"),
)

print("\n=== COALESCE coverage ===")
display(coalesce_coverage)

## EDA: Concept Mapping Preview
Test how well `CodingSchemeCode` values match OMOP `concept` table by vocabulary.

Low match rates may indicate codes need cleaning (stripping dots, formatting differences).

In [ ]:
# Build a distinct set of source codes with their mapped vocabulary_id
df_source_codes = (
    df_detail.filter(
        F.col("CodingScheme").isNotNull() & F.col("CodingSchemeCode").isNotNull()
    )
    .withColumn(
        "omop_vocabulary_id",
        F.when(
            F.upper(F.col("CodingScheme")).like("%ICD-10%")
            | F.upper(F.col("CodingScheme")).like("%ICD10%"),
            F.lit("ICD10CM"),
        )
        .when(
            F.upper(F.col("CodingScheme")).like("%ICD-9%")
            | F.upper(F.col("CodingScheme")).like("%ICD9%"),
            F.lit("ICD9CM"),
        )
        .when(
            F.upper(F.col("CodingScheme")).like("%SNOMED%"),
            F.lit("SNOMED"),
        )
        .otherwise(F.lit(None)),
    )
    .filter(F.col("omop_vocabulary_id").isNotNull())
    .select("CodingScheme", "CodingSchemeCode", "omop_vocabulary_id")
    .distinct()
)

df_concept = spark.table("_exponent.omop.concept")

# Attempt to join source codes to concept table
df_mapped = df_source_codes.join(
    df_concept.alias("c"),
    (F.col("CodingSchemeCode") == F.col("c.concept_code"))
    & (F.col("omop_vocabulary_id") == F.col("c.vocabulary_id")),
    "left",
)

# Summarize match rates by vocabulary
display(
    df_mapped.groupBy("omop_vocabulary_id")
    .agg(
        F.count("*").alias("total_distinct_codes"),
        F.sum(
            F.when(F.col("c.concept_id").isNotNull(), 1).otherwise(0)
        ).alias("matched"),
        F.sum(
            F.when(F.col("c.concept_id").isNull(), 1).otherwise(0)
        ).alias("unmatched"),
        F.round(
            F.sum(F.when(F.col("c.concept_id").isNotNull(), 1).otherwise(0))
            / F.count("*")
            * 100,
            2,
        ).alias("match_pct"),
    )
    .orderBy("omop_vocabulary_id")
)

# Show sample unmatched codes per vocabulary for debugging
print("\n=== Sample unmatched codes (top 10 per vocabulary) ===")
display(
    df_mapped.filter(F.col("c.concept_id").isNull())
    .select("omop_vocabulary_id", "CodingSchemeCode")
    .distinct()
    .limit(30)
)

## EDA: Standard Concept Resolution
ICD codes are non-standard in OMOP. They must map to Standard SNOMED concepts via:

```
source concept_id → concept_relationship (relationship_id = 'Maps to') → standard concept_id
```

Check what percentage of matched source concepts successfully resolve to a standard concept.

In [ ]:
df_concept_rel = spark.table("_exponent.omop.concept_relationship")

# Start from matched source concepts
df_source_matched = df_mapped.filter(F.col("c.concept_id").isNotNull()).select(
    F.col("omop_vocabulary_id"),
    F.col("c.concept_id").alias("source_concept_id"),
    F.col("c.standard_concept"),
)

# Join to concept_relationship for 'Maps to' standard concept
df_std_check = df_source_matched.join(
    df_concept_rel.alias("cr"),
    (F.col("source_concept_id") == F.col("cr.concept_id_1"))
    & (F.col("cr.relationship_id") == "Maps to"),
    "left",
).join(
    df_concept.alias("std"),
    (F.col("cr.concept_id_2") == F.col("std.concept_id"))
    & (F.col("std.standard_concept") == "S")
    & (F.col("std.domain_id") == "Condition"),
    "left",
)

display(
    df_std_check.groupBy("omop_vocabulary_id")
    .agg(
        F.count("*").alias("source_concepts_matched"),
        F.sum(
            F.when(F.col("std.concept_id").isNotNull(), 1).otherwise(0)
        ).alias("has_standard_mapping"),
        F.sum(
            F.when(F.col("std.concept_id").isNull(), 1).otherwise(0)
        ).alias("no_standard_mapping"),
        F.round(
            F.sum(F.when(F.col("std.concept_id").isNotNull(), 1).otherwise(0))
            / F.count("*")
            * 100,
            2,
        ).alias("standard_mapping_pct"),
    )
    .orderBy("omop_vocabulary_id")
)

---
# Transformation

Build the `condition_occurrence` silver table by joining detail + document, mapping codes to OMOP concepts, and resolving standard concepts.

In [ ]:
source = "allscripts_scm"

silver_condition_occurrence = spark.sql(f"""
WITH source_data AS (
    SELECT
        detail.GUID           AS detail_guid,
        detail.ClientGUID,
        detail.CodingScheme,
        detail.CodingSchemeCode,
        detail.KTreeItemName,
        detail.HeadingOne,
        detail.CreatedWhen    AS detail_created,
        doc.AuthoredDtm,
        doc.ServiceDtmUTC,
        doc.Entered           AS doc_entered,
        doc.AuthoredProviderGUID,
        doc.ClientVisitGUID,
        CASE
            WHEN UPPER(detail.CodingScheme) LIKE '%ICD-10%'
              OR UPPER(detail.CodingScheme) LIKE '%ICD10%'
            THEN 'ICD10CM'
            WHEN UPPER(detail.CodingScheme) LIKE '%ICD-9%'
              OR UPPER(detail.CodingScheme) LIKE '%ICD9%'
            THEN 'ICD9CM'
            WHEN UPPER(detail.CodingScheme) LIKE '%SNOMED%'
            THEN 'SNOMED'
            ELSE NULL
        END AS omop_vocabulary_id
    FROM `_exponent`.`_bronze_allscripts_scm_prod_01`.`dbo_cv3clientdocdetail_bkp` detail
    INNER JOIN `_exponent`.`_bronze_allscripts_scm_prod_01`.`dbo_cv3clientdocumentcur` doc
        ON detail.ClientDocumentGUID = doc.GUID
    WHERE detail.Active = TRUE
      AND doc.Active = TRUE
      AND doc.IsCanceled = FALSE
      AND detail.CodingScheme IS NOT NULL
      AND detail.CodingSchemeCode IS NOT NULL
      AND TRIM(detail.CodingSchemeCode) != ''
      AND detail.ClientGUID IS NOT NULL
      AND COALESCE(doc.AuthoredDtm, doc.ServiceDtmUTC, doc.Entered, detail.CreatedWhen) IS NOT NULL
)

SELECT
    -- Standard concept: resolved via concept_relationship 'Maps to'
    COALESCE(std_concept.concept_id, 0) AS condition_concept_id,

    -- Dates
    DATE(COALESCE(sd.AuthoredDtm, sd.ServiceDtmUTC, sd.doc_entered, sd.detail_created))
        AS condition_start_date,
    COALESCE(sd.AuthoredDtm, sd.ServiceDtmUTC, sd.doc_entered, sd.detail_created)
        AS condition_start_datetime,
    NULL AS condition_end_date,
    NULL AS condition_end_datetime,

    -- Type
    32817 AS condition_type_concept_id,

    -- Status
    0 AS condition_status_concept_id,
    NULL AS stop_reason,

    -- Source values
    sd.CodingSchemeCode AS condition_source_value,
    COALESCE(src_concept.concept_id, 0) AS condition_source_concept_id,
    sd.HeadingOne AS condition_status_source_value,

    -- FK source values (resolved to IDs in gold layer)
    CONCAT('{source}', ' | ', CAST(sd.ClientGUID AS STRING)) AS person_source_value,
    CASE
        WHEN sd.AuthoredProviderGUID IS NOT NULL
        THEN CONCAT('{source}', ' | ', CAST(sd.AuthoredProviderGUID AS STRING))
        ELSE NULL
    END AS provider_source_value,
    CASE
        WHEN sd.ClientVisitGUID IS NOT NULL
        THEN CONCAT('{source}', ' | ', CAST(sd.ClientVisitGUID AS STRING))
        ELSE NULL
    END AS visit_occurrence_source_value,
    NULL AS visit_detail_source_value,

    -- Unique identifier for this condition occurrence
    CONCAT('{source}', ' | ', CAST(sd.detail_guid AS STRING)) AS condition_occurrence_source_value,
    '{source}' AS source_system

FROM source_data sd

-- Source concept: match CodingSchemeCode to concept table by vocabulary
LEFT OUTER JOIN `_exponent`.`omop`.`concept` src_concept
    ON src_concept.concept_code = sd.CodingSchemeCode
   AND src_concept.vocabulary_id = sd.omop_vocabulary_id

-- Standard concept: resolve via concept_relationship 'Maps to'
LEFT OUTER JOIN `_exponent`.`omop`.`concept_relationship` cr
    ON cr.concept_id_1 = src_concept.concept_id
   AND cr.relationship_id = 'Maps to'

LEFT OUTER JOIN `_exponent`.`omop`.`concept` std_concept
    ON std_concept.concept_id = cr.concept_id_2
   AND std_concept.standard_concept = 'S'
   AND std_concept.domain_id = 'Condition'

-- Person resolution
INNER JOIN `_exponent`.`omop_mapping`.`source_to_person` stp
    ON stp.person_source_value = CONCAT('{source}', ' | ', CAST(sd.ClientGUID AS STRING))
   AND stp.active_flag = TRUE

WHERE sd.omop_vocabulary_id IS NOT NULL
""")

print(f"Condition occurrence records: {silver_condition_occurrence.count():,}")
display(silver_condition_occurrence)

## Validation Checks

In [ ]:
total_records = silver_condition_occurrence.count()

validation = silver_condition_occurrence.select(
    F.lit(total_records).alias("total_condition_occurrences"),
    F.countDistinct("person_source_value").alias("distinct_patients"),
    F.sum(
        F.when(F.col("condition_start_date").isNull(), 1).otherwise(0)
    ).alias("null_start_dates"),
    F.sum(
        F.when(F.col("condition_source_value").isNull(), 1).otherwise(0)
    ).alias("null_source_values"),
    F.sum(
        F.when(F.col("condition_concept_id") == 0, 1).otherwise(0)
    ).alias("unmapped_concept_id"),
    F.round(
        F.sum(F.when(F.col("condition_concept_id") == 0, 1).otherwise(0))
        / F.lit(total_records)
        * 100,
        2,
    ).alias("unmapped_concept_pct"),
    F.sum(
        F.when(F.col("condition_source_concept_id") == 0, 1).otherwise(0)
    ).alias("unmapped_source_concept_id"),
    F.round(
        F.sum(F.when(F.col("condition_source_concept_id") == 0, 1).otherwise(0))
        / F.lit(total_records)
        * 100,
        2,
    ).alias("unmapped_source_concept_pct"),
    F.sum(
        F.when(F.col("person_source_value").isNull(), 1).otherwise(0)
    ).alias("null_person_source_values"),
    F.min("condition_start_date").alias("min_start_date"),
    F.max("condition_start_date").alias("max_start_date"),
)

display(validation)

# Top 10 most frequent condition_concept_ids with concept names
print("\n=== Top 10 most frequent condition_concept_ids ===")
df_concept = spark.table("_exponent.omop.concept")

display(
    silver_condition_occurrence.filter(F.col("condition_concept_id") != 0)
    .groupBy("condition_concept_id")
    .agg(F.count("*").alias("count"))
    .join(
        df_concept.select("concept_id", "concept_name"),
        F.col("condition_concept_id") == F.col("concept_id"),
        "left",
    )
    .select("condition_concept_id", "concept_name", "count")
    .orderBy(F.desc("count"))
    .limit(10)
)

# Distribution by source vocabulary
print("\n=== Distribution by source vocabulary ===")
display(
    silver_condition_occurrence
    .join(
        df_concept.select(
            F.col("concept_id").alias("src_cid"),
            F.col("vocabulary_id"),
        ),
        F.col("condition_source_concept_id") == F.col("src_cid"),
        "left",
    )
    .groupBy(
        F.coalesce(F.col("vocabulary_id"), F.lit("Unmapped")).alias("source_vocabulary")
    )
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
)

## Select & Create TempView

In [ ]:
silver_condition_occurrence_df = silver_condition_occurrence
silver_condition_occurrence_df.createOrReplaceTempView("silver_condition_occurrence")

---
## Write to Silver Layer

In [ ]:
# -- Merge to Silver layer --
# Uncomment when ready to persist

# spark.sql("""
# MERGE INTO _exponent.omop_silver.condition_occurrence AS t
# USING silver_condition_occurrence AS s
# ON t.condition_occurrence_source_value = s.condition_occurrence_source_value
#
# WHEN MATCHED AND (
#      NOT (t.condition_concept_id <=> s.condition_concept_id)
#   OR NOT (t.condition_start_date <=> s.condition_start_date)
#   OR NOT (t.condition_start_datetime <=> s.condition_start_datetime)
#   OR NOT (t.condition_end_date <=> s.condition_end_date)
#   OR NOT (t.condition_end_datetime <=> s.condition_end_datetime)
#   OR NOT (t.condition_type_concept_id <=> s.condition_type_concept_id)
#   OR NOT (t.condition_status_concept_id <=> s.condition_status_concept_id)
#   OR NOT (t.stop_reason <=> s.stop_reason)
#   OR NOT (t.condition_source_value <=> s.condition_source_value)
#   OR NOT (t.condition_source_concept_id <=> s.condition_source_concept_id)
#   OR NOT (t.condition_status_source_value <=> s.condition_status_source_value)
#   OR NOT (t.person_source_value <=> s.person_source_value)
#   OR NOT (t.provider_source_value <=> s.provider_source_value)
#   OR NOT (t.visit_occurrence_source_value <=> s.visit_occurrence_source_value)
#   OR NOT (t.source_system <=> s.source_system)
# )
# THEN UPDATE SET
#   t.condition_concept_id          = s.condition_concept_id,
#   t.condition_start_date          = s.condition_start_date,
#   t.condition_start_datetime      = s.condition_start_datetime,
#   t.condition_end_date            = s.condition_end_date,
#   t.condition_end_datetime        = s.condition_end_datetime,
#   t.condition_type_concept_id     = s.condition_type_concept_id,
#   t.condition_status_concept_id   = s.condition_status_concept_id,
#   t.stop_reason                   = s.stop_reason,
#   t.condition_source_value        = s.condition_source_value,
#   t.condition_source_concept_id   = s.condition_source_concept_id,
#   t.condition_status_source_value = s.condition_status_source_value,
#   t.person_source_value           = s.person_source_value,
#   t.provider_source_value         = s.provider_source_value,
#   t.visit_occurrence_source_value = s.visit_occurrence_source_value,
#   t.visit_detail_source_value     = s.visit_detail_source_value,
#   t.source_system                 = s.source_system,
#   t.last_mod_tsp                  = current_timestamp()
#
# WHEN NOT MATCHED THEN
# INSERT (
#   condition_concept_id,
#   condition_start_date,
#   condition_start_datetime,
#   condition_end_date,
#   condition_end_datetime,
#   condition_type_concept_id,
#   condition_status_concept_id,
#   stop_reason,
#   condition_source_value,
#   condition_source_concept_id,
#   condition_status_source_value,
#   person_source_value,
#   provider_source_value,
#   visit_occurrence_source_value,
#   visit_detail_source_value,
#   condition_occurrence_source_value,
#   source_system,
#   last_mod_tsp
# )
# VALUES (
#   s.condition_concept_id,
#   s.condition_start_date,
#   s.condition_start_datetime,
#   s.condition_end_date,
#   s.condition_end_datetime,
#   s.condition_type_concept_id,
#   s.condition_status_concept_id,
#   s.stop_reason,
#   s.condition_source_value,
#   s.condition_source_concept_id,
#   s.condition_status_source_value,
#   s.person_source_value,
#   s.provider_source_value,
#   s.visit_occurrence_source_value,
#   s.visit_detail_source_value,
#   s.condition_occurrence_source_value,
#   s.source_system,
#   current_timestamp()
# );
# """)

## Insert Mapping Records

In [ ]:
# -- Insert new mappings to source_to_condition_occurrence --
# Uncomment when ready to persist

# spark.sql("""
# INSERT INTO _exponent.omop_mapping.source_to_condition_occurrence (
#     source_system,
#     condition_occurrence_source_value,
#     active_flag,
#     created_tsp,
#     last_mod_tsp
# )
# SELECT
#     s.source_system,
#     s.condition_occurrence_source_value,
#     TRUE AS active_flag,
#     current_timestamp() AS created_tsp,
#     current_timestamp() AS last_mod_tsp
# FROM (
#     SELECT DISTINCT source_system, condition_occurrence_source_value
#     FROM _exponent.omop_silver.condition_occurrence
#     WHERE source_system = 'allscripts_scm'
# ) s
# LEFT ANTI JOIN _exponent.omop_mapping.source_to_condition_occurrence x
#   ON s.condition_occurrence_source_value = x.condition_occurrence_source_value;
# """)

## Merge to Gold Layer

In [ ]:
# -- Merge to Gold layer --
# Uncomment when ready to persist

# spark.sql("""
# MERGE INTO _exponent.omop.condition_occurrence AS gold
# USING (
#   SELECT
#     sco.condition_occurrence_id,
#     stp.person_id,
#     s.condition_concept_id,
#     s.condition_start_date,
#     s.condition_start_datetime,
#     s.condition_end_date,
#     s.condition_end_datetime,
#     s.condition_type_concept_id,
#     s.condition_status_concept_id,
#     s.stop_reason,
#     NULL AS provider_id,
#     NULL AS visit_occurrence_id,
#     NULL AS visit_detail_id,
#     s.condition_source_value,
#     s.condition_source_concept_id,
#     s.condition_status_source_value
#   FROM _exponent.omop_silver.condition_occurrence s
#   JOIN _exponent.omop_mapping.source_to_condition_occurrence sco
#     ON sco.condition_occurrence_source_value = s.condition_occurrence_source_value
#    AND sco.active_flag = TRUE
#   JOIN _exponent.omop_mapping.source_to_person stp
#     ON stp.person_source_value = s.person_source_value
#    AND stp.active_flag = TRUE
#   WHERE s.source_system = 'allscripts_scm'
# ) AS src
# ON gold.condition_occurrence_id = src.condition_occurrence_id
#
# WHEN MATCHED THEN UPDATE SET
#   gold.person_id                     = src.person_id,
#   gold.condition_concept_id          = src.condition_concept_id,
#   gold.condition_start_date          = src.condition_start_date,
#   gold.condition_start_datetime      = src.condition_start_datetime,
#   gold.condition_end_date            = src.condition_end_date,
#   gold.condition_end_datetime        = src.condition_end_datetime,
#   gold.condition_type_concept_id     = src.condition_type_concept_id,
#   gold.condition_status_concept_id   = src.condition_status_concept_id,
#   gold.stop_reason                   = src.stop_reason,
#   gold.provider_id                   = src.provider_id,
#   gold.visit_occurrence_id           = src.visit_occurrence_id,
#   gold.visit_detail_id              = src.visit_detail_id,
#   gold.condition_source_value        = src.condition_source_value,
#   gold.condition_source_concept_id   = src.condition_source_concept_id,
#   gold.condition_status_source_value = src.condition_status_source_value
#
# WHEN NOT MATCHED THEN INSERT (
#   condition_occurrence_id,
#   person_id,
#   condition_concept_id,
#   condition_start_date,
#   condition_start_datetime,
#   condition_end_date,
#   condition_end_datetime,
#   condition_type_concept_id,
#   condition_status_concept_id,
#   stop_reason,
#   provider_id,
#   visit_occurrence_id,
#   visit_detail_id,
#   condition_source_value,
#   condition_source_concept_id,
#   condition_status_source_value
# )
# VALUES (
#   src.condition_occurrence_id,
#   src.person_id,
#   src.condition_concept_id,
#   src.condition_start_date,
#   src.condition_start_datetime,
#   src.condition_end_date,
#   src.condition_end_datetime,
#   src.condition_type_concept_id,
#   src.condition_status_concept_id,
#   src.stop_reason,
#   src.provider_id,
#   src.visit_occurrence_id,
#   src.visit_detail_id,
#   src.condition_source_value,
#   src.condition_source_concept_id,
#   src.condition_status_source_value
# );
# """)